<a href="https://colab.research.google.com/github/dianasotelo02-crypto/clasificador-fastapi/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Clasificador como servicio web con FastAPI
**Objetivo:** convertir un modelo de clasificación en un servicio web (API REST) que otras aplicaciones puedan consumir, y comprobar que funciona con pruebas y monitoreo.

### ¿Qué es una API REST?
Es un "mesero" entre tu modelo y el resto del mundo: otra aplicación envía una petición HTTP con datos (JSON) y la API responde con la predicción.

```
Cliente (Postman / curl / app)  --JSON-->  FastAPI  -->  Modelo  -->  predicción  --JSON-->  Cliente
```

###Contenido del notebook
1. Preparar el entorno y la estructura del proyecto
2. Entrenar y guardar el modelo
3. Implementar la API en FastAPI
4. Ejecutar el servidor
5. Probar con curl / Python (y Postman)
6. Registro (logging) y monitoreo
7. Pruebas automáticas
8. Conclusiones

## 1.  Entorno y estructura del proyecto
Instalamos las librerías y creamos las carpetas.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" scikit-learn joblib pytest httpx requests
import os
for d in ["app", "tests", "models", "logs", "capturas"]:
    os.makedirs(d, exist_ok=True)
open("app/__init__.py", "a").close()
print("Estructura creada ")

Estructura creada 


## 2. Entrenar y guardar el modelo
Usamos el dataset **Iris** (clasificar flores en *setosa, versicolor, virginica* según 4 medidas) con un **Random Forest**.
Guardamos el modelo con `joblib` para que la API lo cargue sin reentrenar.

In [ ]:
%%writefile train_model.py
"""Entrena un modelo de clasificación (Iris) y lo guarda en disco."""
import json
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy en test: {accuracy:.3f}")

joblib.dump(
    {"model": model, "classes": list(iris.target_names), "accuracy": float(accuracy)},
    "models/iris_model.joblib",
)
print("Modelo guardado en models/iris_model.joblib")

Writing train_model.py


In [ ]:
!python train_model.py

Accuracy en test: 0.900
Modelo guardado en models/iris_model.joblib


## 3. Implementar la API
### 3.1 Esquemas (validación de datos)
Con **Pydantic** definimos qué datos acepta la API. Si alguien envía un valor negativo o falta un campo, FastAPI responde automáticamente con un error **422** sin que el modelo se ejecute.

In [ ]:
%%writefile app/schemas.py
"""Esquemas de entrada/salida. Pydantic valida los datos automáticamente."""
from typing import List
from pydantic import BaseModel, Field


class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., gt=0, le=10, description="Largo del sépalo (cm)", examples=[5.1])
    sepal_width: float = Field(..., gt=0, le=10, description="Ancho del sépalo (cm)", examples=[3.5])
    petal_length: float = Field(..., gt=0, le=10, description="Largo del pétalo (cm)", examples=[1.4])
    petal_width: float = Field(..., gt=0, le=10, description="Ancho del pétalo (cm)", examples=[0.2])


class PredictionResponse(BaseModel):
    prediction: str
    confidence: float
    probabilities: dict[str, float]


class BatchRequest(BaseModel):
    samples: List[IrisFeatures] = Field(..., min_length=1, max_length=100)


class BatchResponse(BaseModel):
    predictions: List[PredictionResponse]

Writing app/schemas.py


### 3.2 Endpoints, logging y métricas
Partes clave del archivo `app/main.py`:

| Parte | Para qué sirve |
|---|---|
| `lifespan` | Carga el modelo **una sola vez** al iniciar (más rápido) |
| `@app.middleware("http")` | Registra **cada petición** (método, ruta, código, tiempo) y cuenta errores |
| `RotatingFileHandler` | Guarda los logs en `logs/api.log` sin crecer indefinidamente |
| `/predict` | Predicción de una muestra |
| `/predict/batch` | Predicción de varias muestras |
| `/health`, `/metrics` | **Monitoreo**: estado del servicio y contadores |

In [ ]:
%%writefile app/main.py
"""API REST con FastAPI para servir un modelo de clasificación."""
import logging
import time
import uuid
from contextlib import asynccontextmanager
from logging.handlers import RotatingFileHandler
from pathlib import Path

import joblib
import numpy as np
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse

from app.schemas import BatchRequest, BatchResponse, IrisFeatures, PredictionResponse

# ---------------------------------------------------------------- LOGGING ---
Path("logs").mkdir(exist_ok=True)
logger = logging.getLogger("api")
logger.setLevel(logging.INFO)
if not logger.handlers:
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    file_handler = RotatingFileHandler("logs/api.log", maxBytes=1_000_000, backupCount=3)
    file_handler.setFormatter(fmt)
    console = logging.StreamHandler()
    console.setFormatter(fmt)
    logger.addHandler(file_handler)
    logger.addHandler(console)

# ------------------------------------------------------- MÉTRICAS SIMPLES ---
metrics = {"requests": 0, "predictions": 0, "errors": 0, "total_time_ms": 0.0}
state: dict = {}
MODEL_PATH = "models/iris_model.joblib"


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Carga el modelo UNA sola vez, al iniciar el servidor."""
    try:
        state["bundle"] = joblib.load(MODEL_PATH)
        logger.info("Modelo cargado correctamente desde %s", MODEL_PATH)
    except Exception:
        logger.exception("No se pudo cargar el modelo")
        state["bundle"] = None
    yield
    logger.info("Servidor detenido")


app = FastAPI(
    title="API de Clasificación de Iris",
    description="Servicio web que predice la especie de una flor Iris.",
    version="1.0.0",
    lifespan=lifespan,
)


# -------------------------------------------------------------- MIDDLEWARE ---
@app.middleware("http")
async def log_requests(request: Request, call_next):
    """Registra cada petición: id, método, ruta, estado y duración."""
    request_id = str(uuid.uuid4())[:8]
    start = time.perf_counter()
    try:
        response = await call_next(request)
    except Exception:
        metrics["errors"] += 1
        logger.exception("[%s] Error no controlado en %s", request_id, request.url.path)
        return JSONResponse(status_code=500, content={"detail": "Error interno del servidor"})
    elapsed = (time.perf_counter() - start) * 1000
    metrics["requests"] += 1
    metrics["total_time_ms"] += elapsed
    if response.status_code >= 400:
        metrics["errors"] += 1
    level = logging.WARNING if response.status_code >= 400 else logging.INFO
    logger.log(level, "[%s] %s %s -> %s (%.1f ms)", request_id, request.method,
               request.url.path, response.status_code, elapsed)
    return response


# --------------------------------------------------------------- UTILIDAD ---
def _predict(samples: list[IrisFeatures]) -> list[PredictionResponse]:
    bundle = state.get("bundle")
    if bundle is None:
        raise HTTPException(status_code=503, detail="Modelo no disponible")
    X = np.array([[s.sepal_length, s.sepal_width, s.petal_length, s.petal_width] for s in samples])
    probs = bundle["model"].predict_proba(X)
    classes = bundle["classes"]
    results = []
    for p in probs:
        idx = int(np.argmax(p))
        results.append(PredictionResponse(
            prediction=classes[idx],
            confidence=round(float(p[idx]), 4),
            probabilities={c: round(float(v), 4) for c, v in zip(classes, p)},
        ))
    metrics["predictions"] += len(results)
    return results


# --------------------------------------------------------------- ENDPOINTS ---
@app.get("/", tags=["General"])
def root():
    return {"mensaje": "API de clasificación activa. Documentación en /docs"}


@app.get("/health", tags=["Monitoreo"])
def health():
    """Indica si el servicio y el modelo están operativos."""
    ok = state.get("bundle") is not None
    return {"status": "ok" if ok else "degradado", "modelo_cargado": ok}


@app.get("/model-info", tags=["General"])
def model_info():
    bundle = state.get("bundle")
    if bundle is None:
        raise HTTPException(status_code=503, detail="Modelo no disponible")
    return {"tipo": type(bundle["model"]).__name__, "clases": bundle["classes"],
            "accuracy_test": bundle["accuracy"]}


@app.get("/metrics", tags=["Monitoreo"])
def get_metrics():
    n = metrics["requests"]
    return {**metrics, "avg_latency_ms": round(metrics["total_time_ms"] / n, 2) if n else 0}


@app.post("/predict", response_model=PredictionResponse, tags=["Predicción"])
def predict(features: IrisFeatures):
    """Predice la especie de UNA flor."""
    result = _predict([features])[0]
    logger.info("Predicción: %s (confianza %.2f)", result.prediction, result.confidence)
    return result


@app.post("/predict/batch", response_model=BatchResponse, tags=["Predicción"])
def predict_batch(batch: BatchRequest):
    """Predice la especie de VARIAS flores en una sola llamada."""
    return BatchResponse(predictions=_predict(batch.samples))

Writing app/main.py


## 4. Ejecutar el servidor
Lo lanzamos en segundo plano para poder seguir usando el notebook.

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print(requests.get("http://127.0.0.1:8000/health").json())

{'status': 'ok', 'modelo_cargado': True}


## 5. Probar la API
### 5.1 Con curl
**Caso 1 – flor tipo *setosa*** (pétalos pequeños):

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'

{"prediction":"setosa","confidence":1.0,"probabilities":{"setosa":1.0,"versicolor":0.0,"virginica":0.0}}

   ![Caso 1: setosa](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/curl_caso1_setosa.PNG)
      La API respondió "setosa" con confianza de 1.0, que era lo esperado porque los pétalos son muy pequeños.

**Caso 2 – flor tipo *virginica*** (pétalos grandes):

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"sepal_length":6.7,"sepal_width":3.0,"petal_length":5.2,"petal_width":2.3}'

{"prediction":"virginica","confidence":1.0,"probabilities":{"setosa":0.0,"versicolor":0.0,"virginica":1.0}}

![Caso 2: virginica](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/curl_caso2_virginica.PNG)
Con medidas de pétalos grandes (largo 5.2 y ancho 2.3), la API respondió “virginica” con confianza de 1.0. Esto muestra que el modelo distingue entre especies según el tamaño de las flores.

**Caso 3 – predicción por lotes** (varias flores en una llamada):

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/predict/batch \
  -H "Content-Type: application/json" \
  -d '{"samples":[{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2},{"sepal_length":5.9,"sepal_width":3.0,"petal_length":4.2,"petal_width":1.5}]}'

{"predictions":[{"prediction":"setosa","confidence":1.0,"probabilities":{"setosa":1.0,"versicolor":0.0,"virginica":0.0}},{"prediction":"versicolor","confidence":0.98,"probabilities":{"setosa":0.0,"versicolor":0.98,"virginica":0.02}}]}

![Caso 3: lote](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/curl_caso3_lote.PNG)
Con el endpoint /predict/batch se enviaron dos flores en una sola petición y la API devolvió una predicción para cada una. Esto sirve cuando otra aplicación necesita clasificar muchos datos sin hacer una llamada por cada uno.

**Caso 4 – datos inválidos** (valor negativo): la API debe rechazarlo con código **422**.

In [ ]:
!curl -s -w "\nCódigo HTTP: %{http_code}\n" -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"sepal_length":-1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'

{"detail":[{"type":"greater_than","loc":["body","sepal_length"],"msg":"Input should be greater than 0","input":-1,"ctx":{"gt":0.0}}]}
Código HTTP: 422


![Caso 4: error 422](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/curl_caso4_error422.PNG)
Se envió un valor negativo en sepal_length para ver cómo reaccionaba la API. Esta rechazó la petición con el código 422 y un mensaje que indica que el valor debe ser mayor que 0. El modelo no llegó a ejecutarse, así que la validación evita que datos incorrectos produzcan predicciones sin sentido.

**Caso 5 – campo faltante:**

In [ ]:
!curl -s -w "\nCódigo HTTP: %{http_code}\n" -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" -d '{"sepal_length":5.1}'

{"detail":[{"type":"missing","loc":["body","sepal_width"],"msg":"Field required","input":{"sepal_length":5.1}},{"type":"missing","loc":["body","petal_length"],"msg":"Field required","input":{"sepal_length":5.1}},{"type":"missing","loc":["body","petal_width"],"msg":"Field required","input":{"sepal_length":5.1}}]}
Código HTTP: 422


![Caso 5: campo faltante](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/curl_caso5_campo_faltante.PNG)
Se envió solo el campo sepal_length, sin los otros tres. La API respondió con el código 422 y el mensaje “Field required” para cada dato que faltaba. Así el servicio exige que la información esté completa antes de predecir.

### 5.2 Con Python (`requests`)
Así otra aplicación podría integrar el servicio:

In [ ]:
import requests
r = requests.post("http://127.0.0.1:8000/predict",
                  json={"sepal_length": 6.0, "sepal_width": 2.9, "petal_length": 4.5, "petal_width": 1.5})
print("Estado:", r.status_code)
print("Respuesta:", r.json())

Estado: 200
Respuesta: {'prediction': 'versicolor', 'confidence': 0.98, 'probabilities': {'setosa': 0.0, 'versicolor': 0.98, 'virginica': 0.02}}


### 5.3 Pruebas con Postman

Además de curl, se probó la API desde Postman, con el servidor corriendo de forma local en el computador (`uvicorn app.main:app --reload`). Postman no se puede conectar a Colab, por eso ess parte se hizo aparte.

**Prueba 1: predicción correcta**

Crear la petición POST a `http://127.0.0.1:8000/predict` y en el Body (raw, JSON) envié las medidas de una flor:

```
{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}
```

La API respondió con código 200 OK y la predicción "setosa", con confianza de 1.0. Era lo esperado, porque los pétalos son muy pequeños.

![Predicción exitosa en Postman](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/postman_predict_ok.PNG)

**Prueba 2: datos inválidos**

Cambiar `sepal_length` por un valor negativo (-1) para ver cómo reaccionaba la API. No ejecutó el modelo y respondió con código 422, indicando que el campo `sepal_length` debe ser mayor que 0.

![Error 422 en Postman](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/postman_error_422.PNG)

**Documentación automática (Swagger)**

También se revisó la página `/docs` que FastAPI genera sola, donde aparecen todos los endpoints y se pueden probar desde el navegador.

![Documentación Swagger](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/swagger_docs.PNG)

Una API no solo debe responder bien con datos correctos, sino también rechazar los incorrectos con un mensaje claro. También se vio que Postman y curl hacen lo mismo, pero Postman es más cómodo para ver la respuesta

## 6. Registro y monitoreo
### 6.1 Logs
Cada petición quedó registrada en `logs/api.log`. Los errores del cliente (4xx) aparecen como `WARNING`:

In [ ]:
!tail -n 15 logs/api.log

2026-10-11 02:59:27,631 | INFO | Modelo cargado correctamente desde models/iris_model.joblib
2026-10-11 02:59:28,467 | INFO | [5d11234e] GET /health -> 200 (5.7 ms)
2026-10-11 02:59:28,585 | INFO | Predicción: setosa (confianza 1.00)
2026-10-11 02:59:28,586 | INFO | [00aaf34a] POST /predict -> 200 (19.7 ms)
2026-10-11 02:59:28,759 | INFO | Predicción: virginica (confianza 1.00)
2026-10-11 02:59:28,760 | INFO | [8d02d88f] POST /predict -> 200 (21.0 ms)
2026-10-11 02:59:28,873 | INFO | [2ca7812b] POST /predict/batch -> 200 (15.3 ms)
2026-10-11 02:59:28,982 | WARNING | [0b7a202c] POST /predict -> 422 (1.1 ms)
2026-10-11 02:59:29,127 | WARNING | [bb936998] POST /predict -> 422 (1.3 ms)
2026-10-11 02:59:29,245 | INFO | Predicción: versicolor (confianza 0.98)
2026-10-11 02:59:29,245 | INFO | [734017a2] POST /predict -> 200 (19.2 ms)


Estos son los registros de la API que corrió en Colab. En el computador también revisé el log del servidor local mientras hacía las pruebas con Postman: la petición exitosa quedó como INFO y la inválida como WARNING con código 422.

![Logs del servidor local](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/logs_servidor.PNG)

### 6.2 Endpoint `/metrics`
Resumen en vivo: total de peticiones, predicciones, errores y latencia media.

In [ ]:
print(requests.get("http://127.0.0.1:8000/metrics").json())
print(requests.get("http://127.0.0.1:8000/model-info").json())

{'requests': 7, 'predictions': 5, 'errors': 2, 'total_time_ms': 83.3981909997874, 'avg_latency_ms': 11.91}
{'tipo': 'RandomForestClassifier', 'clases': ['setosa', 'versicolor', 'virginica'], 'accuracy_test': 0.9}


Con el endpoint `/metrics` se consulto los contadores del servicio desde Postman. Mostró 5 peticiones, 1 predicción y 1 error, que coincide con lo que se hizo: abrir la documentación, una predicción correcta y una petición inválida.

![Métricas del servicio en Postman](https://raw.githubusercontent.com/dianasotelo02-crypto/clasificador-fastapi/main/capturas/postman_metrics.PNG)

### 6.3 Detectar fallos
`/health` sirve para que un sistema externo verifique si el servicio está bien: devuelve `degradado` si el modelo no pudo cargarse. Los fallos al cargar el modelo y los errores internos (500) se registran con la traza completa (`logger.exception`).

## 7. Pruebas automáticas (pytest)
Automatizan las verificaciones anteriores para detectar errores al modificar el código.

In [ ]:
%%writefile tests/test_api.py
"""Pruebas automáticas. Ejecutar con:  pytest -v"""
from fastapi.testclient import TestClient
from app.main import app

VALID = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}


def test_health():
    with TestClient(app) as c:
        r = c.get("/health")
        assert r.status_code == 200 and r.json()["modelo_cargado"] is True


def test_predict_setosa():
    with TestClient(app) as c:
        r = c.post("/predict", json=VALID)
        assert r.status_code == 200
        assert r.json()["prediction"] == "setosa"


def test_predict_virginica():
    with TestClient(app) as c:
        r = c.post("/predict", json={"sepal_length": 6.7, "sepal_width": 3.0,
                                     "petal_length": 5.2, "petal_width": 2.3})
        assert r.json()["prediction"] == "virginica"


def test_batch():
    with TestClient(app) as c:
        r = c.post("/predict/batch", json={"samples": [VALID, VALID]})
        assert r.status_code == 200 and len(r.json()["predictions"]) == 2


def test_invalid_value_returns_422():
    with TestClient(app) as c:
        r = c.post("/predict", json={**VALID, "sepal_length": -1})
        assert r.status_code == 422


def test_missing_field_returns_422():
    with TestClient(app) as c:
        r = c.post("/predict", json={"sepal_length": 5.1})
        assert r.status_code == 422

Writing tests/test_api.py


In [ ]:
open("pytest.ini", "w").write("[pytest]\npythonpath = .\nfilterwarnings = ignore::DeprecationWarning\n")
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
configfile: pytest.ini
plugins: langsmith-0.14.2, typeguard-4.6.0, anyio-4.15.1, platformdirs-4.12.2
collected 6 items                                                              

tests/test_api.py::test_health PASSED                                    [ 16%]
tests/test_api.py::test_predict_setosa PASSED                            [ 33%]
tests/test_api.py::test_predict_virginica PASSED                         [ 50%]
tests/test_api.py::test_batch PASSED                                     [ 66%]
tests/test_api.py::test_invalid_value_returns_422 PASSED                 [ 83%]
tests/test_api.py::test_missing_field_returns_422 PASSED                 [100%]

============================== 6 passed in 5.98s ===============================


## 8. Cierre
Detener el servidor:

In [ ]:
server.terminate()
print("Servidor detenido")

Servidor detenido


##  Conclusiones
- Se entrenó un modelo de clasificación y se expuso como **API REST** con FastAPI.
- La validación con Pydantic evita que datos incorrectos lleguen al modelo (error 422).
- Se verificó el funcionamiento con **curl, Python y Postman**, y con **pytest**.
- El **logging** y el endpoint `/metrics` permiten monitorear el uso y detectar errores.
- **Mejoras futuras:** autenticación con API key, contenedor Docker, despliegue en la nube (Render, Railway, Cloud Run), versionado del modelo y monitoreo con Prometheus/Grafana.

**Repositorio:** `  https://github.com/dianasotelo02-crypto/clasificador-fastapi`